# **PONTIFICIA UNIVERSIDAD JAVERIANA**
## **Procesamiento de alto volumen de datos**
**Fecha:** 7 de Abril del 2026

**Autor:** Grupo Sigma

**Tema:** Proyecto de Big Data

**Objetivo:** 
- Entender la importancia del uso de herramientas de Big Data en entornos empresariales, a fin de poder solucionar preguntas de negocio.
- Entender el paso a paso de un proyecto de procesamiento de datos para la generación de hallazgos de valor basado en la metodología CRISP-DM.
- Documentar la implementación de un cluster como infraestructura de procesamiento de grandes volúmenes de datos, a través de máquinas virtuales.
- Realizar procesamiento de datos aplicado a los resultados de las pruebas Saber 11 del ICFES a nivel nacional.

**Dataset:** Resultados_únicos_Saber_11_20260408.csv  
**Fuente:** Instituto Colombiano para la Evaluación de la Educación (ICFES) — Datos Abiertos  
**Registros:** ~7.11 millones de filas | 51 columnas  

**Version:** Entrega 1

Para asegurar que el proyecto funcione correctamente con pandas, matplotlib, seaborn y findspark, ejecutar el siguiente comando desde la raíz del proyecto
```bash
pip install -r requirements.txt
```

In [1]:
### Importación de bibliotecas basicas 
import os                       # -> Para gestion de archivos y procesos
import sys                      # -> Para manejo de recursos del sistema
import pandas as pd             # -> Para graficar y objetos dataframe
import numpy as np              # -> Para algebra matricial
import matplotlib.pyplot as plt # -> Para formatos de graficas
import seaborn as sns           # -> Para estadistica y graficar
import scipy.stats as stats     # -> Para pruebas estadisticas

In [2]:
### Importacion de bibliotecas especializadas
import findspark                                # -> Para manejo del entorno de PySpark
findspark.init('/Almacen/Spark')                # -> Se inicia el entorno para PySpark
from pyspark import SparkConf, SparkContext     # -> Para contexto y configuración de PySpark
from pyspark.sql import SparkSession            # -> Para manejo de Sesion en entorno de consultas SQL
from pyspark.sql.functions import *             # -> Para funciones de manipulacion de columnas
from pyspark.sql.types import IntegerType, StringType, DoubleType # -> Para definir tipos de datos
import pyspark.sql.functions as F               # -> Para funciones de manipulacion de columnas (alias)
from pyspark.ml.feature import VectorAssembler  # -> Para construcción de vectores  
from pyspark.ml.stat import Correlation         # -> Para calculo de correlaciones

In [3]:
configura = SparkConf()
configura.set('spark.scheduler.mode', 'FAIR')
configura.set('spark.scheduler.allocation', '/Almacen/Spark/conf/fairscheduler.xml')
configura.setMaster('spark://10.43.97.166:7077')
configura.setAppName('SigmaSPARK_ICFES')

sparkSigma = SparkSession.builder.config(conf=configura).getOrCreate()
sparkSigma

## **Lectura de data**

In [4]:
# Lectura del archivo CSV 

dfPy00 = sparkSigma.read.format("csv").option("header","true").load("../../proyecto_procesamiento/data/Resultados_únicos_Saber_11_20260408.csv")
dfPy00.show(5)

+-------+------------------+----------------+-------------------+-------------+---------------+-----------------+-----------------------------+------------------+------------------------+------------------------+-----------------+--------------------+-----------+------------+--------------------+---------------+---------------------------+----------------+-------------------+---------------------------+---------------------------+---------------------+---------------------+-----------------------+-----------------+------------------------+---------------+--------------------+-----------+-----------------------+-----------------+-----------------+----------------+---------------------+-----------------+--------------------+--------------------+--------------------+------------------+-------------------+--------------------+------------------+------------------+-------------+-----------+----------------+------------------------+----------------+--------------------+-----------+
|PERIODO|

### 1 - Descripción general del dataset

El dataset corresponde a los **Resultados únicos de las pruebas Saber 11** publicados por el **ICFES (Instituto Colombiano para la Evaluación de la Educación)**. Contiene información sobre el desempeño académico de estudiantes de grado 11 en Colombia, junto con variables socioeconómicas y de contexto familiar.

**Principales grupos de variables:**
- **Identificación y período:** Año, periodo, número de documento del estudiante.
- **Ubicación geográfica:** Departamento y municipio del colegio y de residencia.
- **Contexto socioeconómico:** Estrato, nivel educativo de los padres, acceso a internet, computador en casa, número de personas en el hogar.
- **Información del colegio:** Nombre, naturaleza (oficial/privado), calendario, área (urbano/rural).
- **Puntajes por área:** Lectura crítica, matemáticas, ciencias naturales, sociales y ciudadanas, inglés.
- **Puntaje global:** Suma ponderada de las áreas evaluadas.

In [5]:
NombresOriginales = [
    # Estudiante
    'ESTU_TIPODOCUMENTO', 'ESTU_CONSECUTIVO', 'ESTU_GENERO', 'ESTU_FECHANACIMIENTO',
    'ESTU_NACIONALIDAD', 'ESTU_PAIS_RESIDE', 'ESTU_PRIVADO_LIBERTAD', 'ESTU_ESTUDIANTE',
    'ESTU_ESTADOINVESTIGACION',
    # Ubicación estudiante
    'ESTU_COD_DEPTO_PRESENTACION', 'ESTU_COD_MCPIO_PRESENTACION',
    'ESTU_COD_RESIDE_DEPTO', 'ESTU_COD_RESIDE_MCPIO',
    'ESTU_DEPTO_PRESENTACION', 'ESTU_MCPIO_PRESENTACION',
    'ESTU_DEPTO_RESIDE', 'ESTU_MCPIO_RESIDE',
    # Colegio - características
    'COLE_AREA_UBICACION', 'COLE_BILINGUE', 'COLE_CALENDARIO', 'COLE_CARACTER',
    'COLE_GENERO', 'COLE_JORNADA', 'COLE_NATURALEZA', 'COLE_SEDE_PRINCIPAL',
    # Colegio - nombres
    'COLE_NOMBRE_ESTABLECIMIENTO', 'COLE_NOMBRE_SEDE',
    # Colegio - códigos
    'COLE_COD_DANE_ESTABLECIMIENTO', 'COLE_COD_DANE_SEDE',
    'COLE_COD_DEPTO_UBICACION', 'COLE_COD_MCPIO_UBICACION', 'COLE_CODIGO_ICFES',
    # Colegio - ubicación
    'COLE_DEPTO_UBICACION', 'COLE_MCPIO_UBICACION',
    # Familia
    'FAMI_ESTRATOVIVIENDA', 'FAMI_TIENEINTERNET', 'FAMI_TIENECOMPUTADOR',
    'FAMI_TIENEAUTOMOVIL', 'FAMI_TIENELAVADORA',
    'FAMI_EDUCACIONPADRE', 'FAMI_EDUCACIONMADRE',
    'FAMI_NUMLIBROS', 'FAMI_CUARTOSHOGAR', 'FAMI_PERSONASHOGAR',
    # Puntajes
    'DESEMP_INGLES', 'PUNT_INGLES', 'PUNT_MATEMATICAS', 'PUNT_C_NATURALES',
    'PUNT_SOCIALES_CIUDADANAS', 'PUNT_LECTURA_CRITICA', 'PUNT_GLOBAL',
    # Periodo
    'PERIODO',
]

NombresNuevos = [
    # Estudiante
    'TIPO_DOC', 'ID_ESTUDIANTE', 'GENERO', 'FECHA_NAC',
    'NACIONALIDAD', 'PAIS', 'PRIVADO_LIBERTAD', 'TIPO_ESTUDIANTE',
    'ESTADO_INVESTIGACION',
    # Ubicación estudiante
    'COD_DEPTO_PRESENTACION', 'COD_MUN_PRESENTACION',
    'COD_DEPTO_RESIDENCIA', 'COD_MUN_RESIDENCIA',
    'DEPTO_PRESENTACION', 'MUN_PRESENTACION',
    'DEPTO_RESIDENCIA', 'MUN_RESIDENCIA',
    # Colegio - características
    'AREA_COLEGIO', 'BILINGUE', 'CALENDARIO', 'CARACTER_COLEGIO',
    'GENERO_COLEGIO', 'JORNADA', 'NATURALEZA', 'SEDE_PRINCIPAL',
    # Colegio - nombres
    'NOMBRE_COLEGIO', 'NOMBRE_SEDE',
    # Colegio - códigos
    'COD_DANE_COLEGIO', 'COD_DANE_SEDE',
    'COD_DEPTO_COLEGIO', 'COD_MUN_COLEGIO', 'COD_ICFES',
    # Colegio - ubicación
    'DEPTO_COLEGIO', 'MUN_COLEGIO',
    # Familia
    'ESTRATO', 'TIENE_INTERNET', 'TIENE_COMPUTADOR',
    'TIENE_AUTOMOVIL', 'TIENE_LAVADORA',
    'EDU_PADRE', 'EDU_MADRE',
    'NUM_LIBROS', 'CUARTOS_HOGAR', 'PERSONAS_HOGAR',
    # Puntajes
    'DESEMP_INGLES', 'PUNT_INGLES', 'PUNT_MATE', 'PUNT_CIENCIAS',
    'PUNT_SOCIALES', 'PUNT_LECTURA', 'PUNT_GLOBAL',
    # Periodo
    'PERIODO',
]

# Aplicar renombres
dfPy01 = dfPy00
for antes, nuevo in zip(NombresOriginales, NombresNuevos):
    if antes in dfPy01.columns:
        dfPy01 = dfPy01.withColumnRenamed(antes, nuevo)

# Verificación
print(f'Columnas en dfPy00: {len(dfPy00.columns)}')
print(f'Columnas en dfPy01: {len(dfPy01.columns)}')
print(f'Columnas sin renombrar (mantienen nombre original):')
sin_renombrar = [c for c in dfPy01.columns if c in dfPy00.columns and c not in NombresNuevos]
print(sin_renombrar if sin_renombrar else 'Ninguna')

Columnas en dfPy00: 51
Columnas en dfPy01: 51
Columnas sin renombrar (mantienen nombre original):
Ninguna


### 2 - Tipos y coherencia de datos

In [6]:
# ── dfPy02: Transformación de tipos de dato ───────────────────────────────────

dfPy02 = dfPy01

# ── 1. Enteros directos (puntajes y período) ──────────────────────────────────

COLS_INT = ['PUNT_INGLES', 'PUNT_MATE', 'PUNT_CIENCIAS',
            'PUNT_SOCIALES', 'PUNT_LECTURA', 'PUNT_GLOBAL', 'PERIODO']

for c in COLS_INT:
    if c in dfPy02.columns:
        dfPy02 = dfPy02.withColumn(c, F.col(c).cast('int'))

# ── 2. Fecha ──────────────────────────────────────────────────────────────────

dfPy02 = dfPy02.withColumn('FECHA_NAC',
    F.to_date(F.col('FECHA_NAC'), 'dd/MM/yyyy')
)

# ── 3. ESTRATO: extraer número de "Estrato 1", "Estrato 2", etc. ──────────────

dfPy02 = dfPy02.withColumn('ESTRATO',
    F.regexp_extract(F.col('ESTRATO'), r'(\d+)', 1).cast('int')
)

# ── 4. CUARTOS_HOGAR: texto → double ─────────────────────────────────────────

def mapear_cuartos(col_name):
    c = F.upper(F.trim(F.col(col_name)))
    return (
        F.when(c == 'UNO',                          1.0)
         .when(c == 'DOS',                          2.0)
         .when(c == 'TRES',                         3.0)
         .when(c == 'CUATRO',                       4.0)
         .when(c == 'CINCO',                        5.0)
         .when(c == 'SEIS',                         6.0)
         .when(c == 'SIETE',                        7.0)
         .when(c == 'OCHO',                         8.0)
         .when(c == 'NUEVE',                        9.0)
         .when(c.isin('SEIS O MAS', 'SEIS O MÁS'), 6.0)
         .when(c.isin('DIEZ O MÁS', 'DIEZ O MAS'), 10.0)
         .otherwise(F.lit(None).cast('double'))
    )

dfPy02 = dfPy02.withColumn('CUARTOS_HOGAR', mapear_cuartos('CUARTOS_HOGAR'))

# ── 5. PERSONAS_HOGAR: texto y rangos → double (punto medio para rangos) ──────

def mapear_personas(col_name):
    c = F.upper(F.trim(F.col(col_name)))
    return (
        F.when(c == 'UNA',                          1.0)
         .when(c == 'DOS',                          2.0)
         .when(c == 'TRES',                         3.0)
         .when(c == 'CUATRO',                       4.0)
         .when(c == 'CINCO',                        5.0)
         .when(c == 'SEIS',                         6.0)
         .when(c == 'SIETE',                        7.0)
         .when(c == 'OCHO',                         8.0)
         .when(c == 'NUEVE',                        9.0)
         .when(c == 'DIEZ',                        10.0)
         .when(c == 'ONCE',                        11.0)
         .when(c.isin('DOCE O MÁS', 'DOCE O MAS'), 12.0)
         .when(c == '1 A 2',                        1.5)
         .when(c == '3 A 4',                        3.5)
         .when(c == '5 A 6',                        5.5)
         .when(c == '7 A 8',                        7.5)
         .when(c.isin('9 O MÁS', '9 O MAS'),        9.0)
         .otherwise(F.lit(None).cast('double'))
    )

dfPy02 = dfPy02.withColumn('PERSONAS_HOGAR', mapear_personas('PERSONAS_HOGAR'))

# ── 6. Estandarización de texto en categóricas (upper + trim) ─────────────────

COLS_STRING = [
    # Estudiante
    'GENERO', 'NACIONALIDAD', 'PAIS', 'PRIVADO_LIBERTAD',
    'TIPO_ESTUDIANTE', 'ESTADO_INVESTIGACION', 'TIPO_DOC',
    # Ubicación
    'DEPTO_PRESENTACION', 'MUN_PRESENTACION', 'DEPTO_RESIDENCIA', 'MUN_RESIDENCIA',
    # Colegio
    'AREA_COLEGIO', 'BILINGUE', 'CALENDARIO', 'CARACTER_COLEGIO',
    'GENERO_COLEGIO', 'JORNADA', 'NATURALEZA', 'SEDE_PRINCIPAL',
    'DEPTO_COLEGIO', 'MUN_COLEGIO',
    # Familia
    'TIENE_INTERNET', 'TIENE_COMPUTADOR', 'TIENE_AUTOMOVIL', 'TIENE_LAVADORA',
    'EDU_PADRE', 'EDU_MADRE', 'NUM_LIBROS',
    # Puntaje inglés
    'DESEMP_INGLES',
]

for c in COLS_STRING:
    if c in dfPy02.columns:
        dfPy02 = dfPy02.withColumn(c, F.upper(F.trim(F.col(c))))

# ── Verificación ──────────────────────────────────────────────────────────────

print('Schema final dfPy02:')
dfPy02.printSchema()

Schema final dfPy02:
root
 |-- PERIODO: integer (nullable = true)
 |-- TIPO_DOC: string (nullable = true)
 |-- ID_ESTUDIANTE: string (nullable = true)
 |-- AREA_COLEGIO: string (nullable = true)
 |-- BILINGUE: string (nullable = true)
 |-- CALENDARIO: string (nullable = true)
 |-- CARACTER_COLEGIO: string (nullable = true)
 |-- COD_DANE_COLEGIO: string (nullable = true)
 |-- COD_DANE_SEDE: string (nullable = true)
 |-- COD_DEPTO_COLEGIO: string (nullable = true)
 |-- COD_MUN_COLEGIO: string (nullable = true)
 |-- COD_ICFES: string (nullable = true)
 |-- DEPTO_COLEGIO: string (nullable = true)
 |-- GENERO_COLEGIO: string (nullable = true)
 |-- JORNADA: string (nullable = true)
 |-- MUN_COLEGIO: string (nullable = true)
 |-- NATURALEZA: string (nullable = true)
 |-- NOMBRE_COLEGIO: string (nullable = true)
 |-- NOMBRE_SEDE: string (nullable = true)
 |-- SEDE_PRINCIPAL: string (nullable = true)
 |-- COD_DEPTO_PRESENTACION: string (nullable = true)
 |-- COD_MUN_PRESENTACION: string (nullab

### **Analisis de nulos**

Tras ejecutar el diagnóstico de frecuencias y conteo de vacíos, se observan los siguientes hallazgos:
- **Género:** Variable con cobertura casi completa. Los registros nulos representan una fracción mínima y serán eliminados para no afectar los análisis demográficos.
- **Puntajes (PUNT_*):** Las columnas de puntajes son las variables objetivo del análisis. Los valores nulos corresponden a presentaciones incompletas o inválidas, y serán excluidos del análisis de rendimiento.
- **Educación de padres (EDU_PADRE, EDU_MADRE):** Variables con porcentaje relevante de valores nulos o categoría `No sabe`. Son importantes para el análisis socioeconómico.
- **Estrato:** Presenta valores nulos y posibles inconsistencias en la codificación (valores fuera del rango 1-6). Se tratarán como categoría especial `SIN_DATO`.
- **Internet y Computador:** Variables binarias con posibles nulos, relevantes para medir la brecha de conectividad digital.
- **Duplicados:** Se detectarán registros duplicados para aplicar la función `.distinct()` antes del modelado.

### **Decisiones de Tratamiento**

- **Limpieza de duplicados:** Se aplicará `.distinct()` para conservar una única presentación por estudiante por período.
- **Puntajes nulos:** Se eliminarán los registros donde `PUNT_GLOBAL` sea nulo, ya que representan presentaciones incompletas o inválidas.
- **Género nulo:** Se eliminarán los 105 registros con género NULL dada su irrelevancia estadística.
- **Estrato:** Los valores fuera del rango 1-6 se reclasificarán como `SIN_DATO` para no distorsionar el análisis socioeconómico.
- **Consolidación de Categorías (Educación de padres):** Se implementará un mapeo para unificar las categorías similares en etiquetas estándar.
- **Exclusión de Variables Incompletas:** Las columnas `PAIS` y `FECHA_NAC` serán descartadas de los análisis predictivos por su bajo aporte informativo.

## **Planteamiento de sub-preguntas investigativas**

1. ¿Existe una brecha significativa en el puntaje global Saber 11 entre colegios oficiales y no oficiales, y cómo varía esta brecha según el departamento?
2. ¿Qué tan fuerte es la relación entre el nivel educativo de los padres (EDU_PADRE, EDU_MADRE) y el rendimiento académico del estudiante en cada área evaluada?
3. ¿Qué variables socioeconómicas (estrato, acceso a internet, computador en casa, número de personas en el hogar) son los mejores predictores del puntaje global en las pruebas Saber 11?
4. ¿Existen diferencias estadísticamente significativas en el rendimiento por área (matemáticas, inglés, ciencias) entre colegios urbanos y rurales a nivel nacional?

## **Limpieza, filtro y transformaciones iniciales**

In [7]:
# Paso 1: Eliminación de duplicados
dfPy03 = dfPy02.distinct()
print('Registros tras eliminar duplicados:', dfPy03.count())

Registros tras eliminar duplicados: 5722593


In [8]:
# Paso 2: Eliminar registros sin PUNT_GLOBAL (presentaciones incompletas)
dfPy03 = dfPy03.filter(F.col('PUNT_GLOBAL').isNotNull())
print('Registros tras eliminar nulos en PUNT_GLOBAL:', dfPy03.count())

Registros tras eliminar nulos en PUNT_GLOBAL: 3395834


In [9]:
# Paso 3: Eliminar registros sin GENERO
dfPy03 = dfPy03.filter(F.col('GENERO').isNotNull())
print('Registros tras eliminar nulos en GENERO:', dfPy03.count())

Registros tras eliminar nulos en GENERO: 3392749


In [10]:
# Filtrar solo registros post-2014, momento en el que se formulo el estandar actual de calificación del ICFES
dfPy03 = dfPy03.filter(F.col('PERIODO').substr(1, 4).cast('int') >= 2014)

print('Registros post-2014:', dfPy03.count())

Registros post-2014: 3392749


In [11]:
# Paso 4: Descarte de columnas con alta ausencia o bajo aporte informativo
columnas_a_descartar = ['PAIS', 'FECHA_NAC', 'TIPO_DOC']
dfPy03 = dfPy03.drop(*[c for c in columnas_a_descartar if c in dfPy03.columns])
print('Columnas restantes:', dfPy03.columns)

Columnas restantes: ['PERIODO', 'ID_ESTUDIANTE', 'AREA_COLEGIO', 'BILINGUE', 'CALENDARIO', 'CARACTER_COLEGIO', 'COD_DANE_COLEGIO', 'COD_DANE_SEDE', 'COD_DEPTO_COLEGIO', 'COD_MUN_COLEGIO', 'COD_ICFES', 'DEPTO_COLEGIO', 'GENERO_COLEGIO', 'JORNADA', 'MUN_COLEGIO', 'NATURALEZA', 'NOMBRE_COLEGIO', 'NOMBRE_SEDE', 'SEDE_PRINCIPAL', 'COD_DEPTO_PRESENTACION', 'COD_MUN_PRESENTACION', 'COD_DEPTO_RESIDENCIA', 'COD_MUN_RESIDENCIA', 'DEPTO_PRESENTACION', 'DEPTO_RESIDENCIA', 'ESTADO_INVESTIGACION', 'TIPO_ESTUDIANTE', 'GENERO', 'MUN_PRESENTACION', 'MUN_RESIDENCIA', 'NACIONALIDAD', 'PRIVADO_LIBERTAD', 'CUARTOS_HOGAR', 'EDU_MADRE', 'EDU_PADRE', 'ESTRATO', 'PERSONAS_HOGAR', 'TIENE_AUTOMOVIL', 'TIENE_COMPUTADOR', 'TIENE_INTERNET', 'TIENE_LAVADORA', 'DESEMP_INGLES', 'PUNT_INGLES', 'PUNT_MATE', 'PUNT_SOCIALES', 'PUNT_CIENCIAS', 'PUNT_LECTURA', 'PUNT_GLOBAL']


In [12]:
# Paso 5: Ver todos los valores en ESTRATO
dfPy03.groupby('ESTRATO').count().orderBy('count', ascending=False).show(20, truncate=False)

+-------+-------+
|ESTRATO|count  |
+-------+-------+
|1      |1209264|
|2      |1140572|
|3      |626700 |
|NULL   |159458 |
|4      |156743 |
|5      |61777  |
|6      |38235  |
+-------+-------+



In [13]:
# Reclasificación del Estrato — valores fuera del rango 1-6 marcados como SIN_DATO
dfPy03 = dfPy03.withColumn('ESTRATO',
    F.when(F.col('ESTRATO').between(1, 6), F.col('ESTRATO').cast('string'))
     .otherwise(F.lit('SIN_DATO'))
)

print('Categorías de ESTRATO tras reclasificación:')
dfPy03.groupby('ESTRATO').count().orderBy(F.col('count').desc()).show()

Categorías de ESTRATO tras reclasificación:
+--------+-------+
| ESTRATO|  count|
+--------+-------+
|       1|1209264|
|       2|1140572|
|       3| 626700|
|SIN_DATO| 159458|
|       4| 156743|
|       5|  61777|
|       6|  38235|
+--------+-------+



# Clusters de estudiantes

Se buscará formar grupos de estudiantes diferentes que permitan conocer agrupaciónes que expliquen los tipos de estudiantes que presentan la prueba ICFES, con el fin de entender y desarrollar planes de acción dedicadas a estos grupos diferentes.

## Tratamiento de nulos en variables categóricas

Los nulos en variables socioeconómicas y de contexto se reemplazan por la categoría `"Sin dato"` para preservar todos los registros válidos en el clustering. Esta decisión evita sesgo por imputación en variables con ausencia informativa (por ejemplo, el estudiante o la familia no proporcionó el dato).

In [14]:
COLS_SIN_DATO = [
    # Recursos del hogar
    'TIENE_INTERNET', 'TIENE_COMPUTADOR', 'TIENE_AUTOMOVIL', 'TIENE_LAVADORA',
    # Educación padres
    'EDU_PADRE', 'EDU_MADRE',
    # Colegio
    'JORNADA', 'AREA_COLEGIO', 'NATURALEZA', 'CALENDARIO',
    'BILINGUE', 'CARACTER_COLEGIO', 'GENERO_COLEGIO',
    # Inglés
    'DESEMP_INGLES',
]

for col_name in COLS_SIN_DATO:
    if col_name in dfPy03.columns:
        dfPy03 = dfPy03.withColumn(
            col_name,
            F.when(F.col(col_name).isNull(), 'Sin dato').otherwise(F.col(col_name))
        )

# Verificación
print('Nulos restantes en variables tratadas:')
dfPy03.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in COLS_SIN_DATO
]).show()

Nulos restantes en variables tratadas:
+--------------+----------------+---------------+--------------+---------+---------+-------+------------+----------+----------+--------+----------------+--------------+-------------+
|TIENE_INTERNET|TIENE_COMPUTADOR|TIENE_AUTOMOVIL|TIENE_LAVADORA|EDU_PADRE|EDU_MADRE|JORNADA|AREA_COLEGIO|NATURALEZA|CALENDARIO|BILINGUE|CARACTER_COLEGIO|GENERO_COLEGIO|DESEMP_INGLES|
+--------------+----------------+---------------+--------------+---------+---------+-------+------------+----------+----------+--------+----------------+--------------+-------------+
|             0|               0|              0|             0|        0|        0|      0|           0|         0|         0|       0|               0|             0|            0|
+--------------+----------------+---------------+--------------+---------+---------+-------+------------+----------+----------+--------+----------------+--------------+-------------+



## Discretización de variables numéricas

Se convierten las variables numéricas a categorías ordinales para unificar el espacio de features antes del clustering con K-modes. Las decisiones de corte son:

| Variable | Criterio de corte | Niveles |
|---|---|---|
| `PUNT_MATE`, `PUNT_LECTURA`, `PUNT_CIENCIAS`, `PUNT_SOCIALES` | Niveles oficiales ICFES (0–40 / 41–55 / 56–70 / 71–100) | Insuficiente · Mínimo · Satisfactorio · Avanzado |
| `PUNT_INGLES` | Marco Común Europeo vía ICFES (0–47 / 48–57 / 58–67 / 68–78 / 79–100) | Pre-A1 · A1 · A2 · B1 · B+ |
| `PUNT_GLOBAL` | Cuartiles calculados por `PERIODO` | Bajo · Medio-bajo · Medio-alto · Alto |
| `HACINAMIENTO_CAT` | Varible creada con los terciales empíricos de `PERSONAS_HOGAR / CUARTOS_HOGAR` | Bajo · Medio · Alto · Sin dato |

`HACINAMIENTO_CAT` es una nueva variable creada que mide la cantidad de personas por cuartos en el hogar, busca resumir ambas variables y dar una tasa que según los análisis realizados en el cuaderno de pobreza, logra explicar con gran precisión el estrato y el nivel de pobreza del estudiante.

Los cortes para las áreas e inglés son los reportados públicamente por el ICFES. Se recomienda verificarlos contra el documento técnico oficial de tu cohorte en caso de que el dataset incluya periodos anteriores a 2015.

Las columnas numéricas originales (`PUNT_*`, `PERSONAS_HOGAR`, `CUARTOS_HOGAR`) se conservan en el DataFrame para validación y perfilamiento posterior.

In [15]:
print(dfPy03.columns)

['PERIODO', 'ID_ESTUDIANTE', 'AREA_COLEGIO', 'BILINGUE', 'CALENDARIO', 'CARACTER_COLEGIO', 'COD_DANE_COLEGIO', 'COD_DANE_SEDE', 'COD_DEPTO_COLEGIO', 'COD_MUN_COLEGIO', 'COD_ICFES', 'DEPTO_COLEGIO', 'GENERO_COLEGIO', 'JORNADA', 'MUN_COLEGIO', 'NATURALEZA', 'NOMBRE_COLEGIO', 'NOMBRE_SEDE', 'SEDE_PRINCIPAL', 'COD_DEPTO_PRESENTACION', 'COD_MUN_PRESENTACION', 'COD_DEPTO_RESIDENCIA', 'COD_MUN_RESIDENCIA', 'DEPTO_PRESENTACION', 'DEPTO_RESIDENCIA', 'ESTADO_INVESTIGACION', 'TIPO_ESTUDIANTE', 'GENERO', 'MUN_PRESENTACION', 'MUN_RESIDENCIA', 'NACIONALIDAD', 'PRIVADO_LIBERTAD', 'CUARTOS_HOGAR', 'EDU_MADRE', 'EDU_PADRE', 'ESTRATO', 'PERSONAS_HOGAR', 'TIENE_AUTOMOVIL', 'TIENE_COMPUTADOR', 'TIENE_INTERNET', 'TIENE_LAVADORA', 'DESEMP_INGLES', 'PUNT_INGLES', 'PUNT_MATE', 'PUNT_SOCIALES', 'PUNT_CIENCIAS', 'PUNT_LECTURA', 'PUNT_GLOBAL']


In [16]:
from pyspark.sql import Window

# ── Cortes oficiales ICFES ────────────────────────────────────────────────────
CORTES_AREA = {
    'PUNT_MATE':     (40, 55, 70),
    'PUNT_LECTURA':  (40, 55, 70),
    'PUNT_CIENCIAS': (40, 55, 70),
    'PUNT_SOCIALES': (40, 55, 70),
}
ETIQUETAS_AREA   = ['1_Insuficiente', '2_Minimo', '3_Satisfactorio', '4_Avanzado']
CORTES_INGLES    = (47, 57, 67, 78)
ETIQUETAS_INGLES = ['1_PreA1', '2_A1', '3_A2', '4_B1', '5_Bmas']

# ── HACINAMIENTO_RAW ──────────────────────────────────────────────────────────
dfPy04 = dfPy03.withColumn(
    'HACINAMIENTO_RAW',
    F.when(
        F.col('CUARTOS_HOGAR').isNotNull() & (F.col('CUARTOS_HOGAR') > 0) & F.col('PERSONAS_HOGAR').isNotNull(),
        F.col('PERSONAS_HOGAR') / F.col('CUARTOS_HOGAR')
    ).otherwise(F.lit(None).cast('double'))
)

# ── HACINAMIENTO_CAT ──────────────────────────────────────────────────────────
q1, q2 = dfPy04.filter(
    F.col('HACINAMIENTO_RAW').isNotNull()
).approxQuantile('HACINAMIENTO_RAW', [1/3, 2/3], 0.01)

print(f'Cortes hacinamiento — Q1: {q1:.2f} | Q2: {q2:.2f}')

dfPy04 = dfPy04.withColumn(
    'HACINAMIENTO_CAT',
    F.when(F.col('HACINAMIENTO_RAW').isNull(), 'Sin_dato')
     .when(F.col('HACINAMIENTO_RAW') <= q1,    '1_Bajo')
     .when(F.col('HACINAMIENTO_RAW') <= q2,    '2_Medio')
     .otherwise('3_Alto')
).drop('HACINAMIENTO_RAW')

# ── Puntajes por área → niveles ICFES ────────────────────────────────────────
for col_nombre, (c1, c2, c3) in CORTES_AREA.items():
    nivel_col = 'NIVEL_' + col_nombre.replace('PUNT_', '')
    dfPy04 = dfPy04.withColumn(
        nivel_col,
        F.when(F.col(col_nombre).isNull(),  'Sin_dato')
         .when(F.col(col_nombre) <= c1,     ETIQUETAS_AREA[0])
         .when(F.col(col_nombre) <= c2,     ETIQUETAS_AREA[1])
         .when(F.col(col_nombre) <= c3,     ETIQUETAS_AREA[2])
         .otherwise(                        ETIQUETAS_AREA[3])
    )

# ── PUNT_INGLES → niveles MCER ────────────────────────────────────────────────
c1, c2, c3, c4 = CORTES_INGLES
dfPy04 = dfPy04.withColumn(
    'NIVEL_INGLES',
    F.when(F.col('PUNT_INGLES').isNull(),  'Sin_dato')
     .when(F.col('PUNT_INGLES') <= c1,     ETIQUETAS_INGLES[0])
     .when(F.col('PUNT_INGLES') <= c2,     ETIQUETAS_INGLES[1])
     .when(F.col('PUNT_INGLES') <= c3,     ETIQUETAS_INGLES[2])
     .when(F.col('PUNT_INGLES') <= c4,     ETIQUETAS_INGLES[3])
     .otherwise(                           ETIQUETAS_INGLES[4])
)

print('Discretización de áreas completada.')

Cortes hacinamiento — Q1: 1.33 | Q2: 1.83
Discretización de áreas completada.


In [17]:
# ── PUNT_GLOBAL → cuartiles por PERIODO ──────────────────────────────────────
window_periodo = Window.partitionBy('PERIODO').orderBy('PUNT_GLOBAL')

MAPA_GLOBAL = {'1': '1_Bajo', '2': '2_Medio_bajo',
               '3': '3_Medio_alto', '4': '4_Alto'}

dfPy04 = dfPy04.withColumn(
    'NIVEL_GLOBAL',
    F.ntile(4).over(window_periodo).cast('string')
)

for num, etiqueta in MAPA_GLOBAL.items():
    dfPy04 = dfPy04.withColumn(
        'NIVEL_GLOBAL',
        F.when(F.col('NIVEL_GLOBAL') == num, etiqueta)
         .otherwise(F.col('NIVEL_GLOBAL'))
    )

print('NIVEL_GLOBAL completado.')

NIVEL_GLOBAL completado.


In [18]:
COLS_DISCRETIZADAS = [
    'HACINAMIENTO_CAT', 'NIVEL_MATE', 'NIVEL_LECTURA',
    'NIVEL_CIENCIAS', 'NIVEL_SOCIALES', 'NIVEL_INGLES', 'NIVEL_GLOBAL'
]

for col_name in COLS_DISCRETIZADAS:
    print(f'─── {col_name} ───')
    dfPy04.groupBy(col_name).count() \
          .orderBy(col_name) \
          .show(truncate=False)

─── HACINAMIENTO_CAT ───
+----------------+-------+
|HACINAMIENTO_CAT|count  |
+----------------+-------+
|1_Bajo          |1204031|
|2_Medio         |1184793|
|3_Alto          |936945 |
|Sin_dato        |66980  |
+----------------+-------+

─── NIVEL_MATE ───
+---------------+-------+
|NIVEL_MATE     |count  |
+---------------+-------+
|1_Insuficiente |688892 |
|2_Minimo       |1578929|
|3_Satisfactorio|945654 |
|4_Avanzado     |179274 |
+---------------+-------+

─── NIVEL_LECTURA ───
+---------------+-------+
|NIVEL_LECTURA  |count  |
+---------------+-------+
|1_Insuficiente |458696 |
|2_Minimo       |1690511|
|3_Satisfactorio|1112714|
|4_Avanzado     |130828 |
+---------------+-------+

─── NIVEL_CIENCIAS ───
+---------------+-------+
|NIVEL_CIENCIAS |count  |
+---------------+-------+
|1_Insuficiente |600017 |
|2_Minimo       |1731810|
|3_Satisfactorio|949300 |
|4_Avanzado     |111622 |
+---------------+-------+

─── NIVEL_SOCIALES ───
+---------------+-------+
|NIVEL_SOCIALES |c

## Selección de variables para el modelo

Las variables se dividen en dos grupos, variables para modelado y perfilamiento. Solo las del modelo entran al algoritmo K-modes; las de perfilamiento se usan después para describir e interpretar cada cluster.

**Criterios de exclusión del modelo:**
- Identificadores y códigos (no aportan distancia semántica)
- Variables geográficas de alta cardinalidad (se usan para mapear clusters después)
- Variables redundantes con otras ya incluidas
- Variables con información posterior al evento (puntajes originales ya resumidos en NIVEL_*)

In [21]:
# ── Variables que entran al modelo K-modes ────────────────────────────────────
MODEL_COLS = [
    # Desempeño académico (discretizado)
    'NIVEL_MATE', 'NIVEL_LECTURA', 'NIVEL_CIENCIAS',
    'NIVEL_SOCIALES', 'NIVEL_INGLES', 'NIVEL_GLOBAL',
    # Contexto socioeconómico del hogar
    'ESTRATO', 'HACINAMIENTO_CAT',
    'TIENE_INTERNET', 'TIENE_COMPUTADOR',
    'EDU_MADRE', 'EDU_PADRE',
    # Contexto del colegio
    'NATURALEZA', 'AREA_COLEGIO', 'JORNADA',
]

# ── Variables reservadas para perfilamiento posterior ─────────────────────────
PROFILING_COLS = [
    # Identificación
    'ID_ESTUDIANTE', 'PERIODO',
    # Geografía
    'DEPTO_COLEGIO', 'MUN_COLEGIO',
    'DEPTO_PRESENTACION', 'DEPTO_RESIDENCIA',
    # Demográficas
    'GENERO', 'FECHA_NAC',
    # Colegio
    'NOMBRE_COLEGIO', 'CALENDARIO', 'BILINGUE',
    'CARACTER_COLEGIO', 'GENERO_COLEGIO',
    # Recursos del hogar adicionales
    'TIENE_AUTOMOVIL', 'TIENE_LAVADORA',
    # Puntajes originales (para validación)
    'PUNT_MATE', 'PUNT_LECTURA', 'PUNT_CIENCIAS',
    'PUNT_SOCIALES', 'PUNT_INGLES', 'PUNT_GLOBAL',
    # Inglés oficial (redundante con NIVEL_INGLES en el modelo)
    'DESEMP_INGLES',
]

In [23]:
# DataFrame reducido para clustering — solo variables del modelo
dfModel = dfPy04.select(['ID_ESTUDIANTE'] + MODEL_COLS)

## Selección de Medida de similitud: Distancia de Hamming en K-modes

### ¿Cómo funciona Hamming en K-modes?

A diferencia de K-means (que opera sobre datos numéricos usando distancia euclídea), **K-modes usa la distancia de Hamming** para comparar registros categóricos. La distancia entre dos registros se define como el número de atributos en los que difieren:

$$d(A, B) = \sum_{j=1}^{p} \delta(a_j, b_j) \quad \text{donde} \quad \delta(a_j, b_j) = \begin{cases} 0 & \text{si } a_j = b_j \\ 1 & \text{si } a_j \neq b_j \end{cases}$$

### Ejemplo con datos reales del dataset

| Variable | Estudiante A | Estudiante B | ¿Iguales? |
|---|---|---|---|
| NIVEL_MATE | 2_Minimo | 2_Minimo | ✅ 0 |
| NIVEL_INGLES | 1_PreA1 | 3_A2 | ❌ 1 |
| ESTRATO | 1 | 6 | ❌ 1 |
| HACINAMIENTO_CAT | 3_Alto | 3_Alto | ✅ 0 |
| TIENE_INTERNET | SI | NO | ❌ 1 |
| NATURALEZA | OFICIAL | OFICIAL | ✅ 0 |
| ... | ... | ... | ... |

**Distancia total = 3** (de 15 atributos posibles → distancia normalizada = 0.20)

### ¿Cómo actualiza K-modes los centroides?

En K-means el centroide es la **media** del cluster. En K-modes el centroide es la **moda** — el valor más frecuente de cada variable dentro del cluster. Esto preserva valores categóricos válidos en los centroides.

### Beneficios y limitaciones

Hamming es una distancia muy sencilla computacionalmente de calcular, siendo más eficiente computacionalmente que otras como Jaccard o Gower, la que la vuelve de las mejores para trabajar grandes volumenes de datos. Hamming trata todas las variables como **nominales puras**: la distancia entre Estrato 1 y Estrato 2 es igual a la distancia entre Estrato 1 y Estrato 6 (ambas = 1). El orden ordinal de variables como `ESTRATO`, `NIVEL_MATE` o `HACINAMIENTO_CAT` no se respeta. Esto debe tenerse en cuenta al interpretar los clusters resultantes.


## Algoritmo: K-modes

Se usa **K-modes** con las siguientes decisiones:

- **Inicialización:** método de Cao — más estable que la inicialización aleatoria, reduce sensibilidad al punto de partida.
- **Réplicas por k:** 10 ejecuciones por cada valor de k para reducir varianza.
- **Rango de k evaluado:** 3 a 6 clusters.
- **Estrategia de ejecución:** dado el volumen (4.5M registros), se trabaja con una **muestra estratificada** para encontrar k óptimo y entrenar el modelo. Luego se asignan los clusters al dataset completo usando los centroides resultantes.

Como primer paso se va a seleccionar una muestra de la base completa para calcular el número de clusters óptimo para el algoritmo, puesto que se desea realizar un análisis de este hiperparámetro, trabajar con los más de 4 millones de registros conllevaría un tiempo computacional extremadamente alto.

In [ ]:
from kmodes.kmodes import KModes
import pandas as pd
import numpy as np

# ── Muestra estratificada por NIVEL_GLOBAL y NATURALEZA ──────────────────────
# Garantiza representación de todos los perfiles de rendimiento y tipo de colegio

fraccion = 0.10  # 10% ≈ 338k registros — suficiente para estabilidad de K-modes

pdf_sample = dfModel.sampleBy(
    'NIVEL_GLOBAL',
    fractions={
        '1_Bajo':       fraccion,
        '2_Medio_bajo': fraccion,
        '3_Medio_alto': fraccion,
        '4_Alto':       fraccion,
    },
    seed=42
).drop('ID_ESTUDIANTE').toPandas()

print(f'Tamaño de la muestra: {len(pdf_sample):,} registros')
print(f'Columnas: {list(pdf_sample.columns)}')

### Selección del número de clusters

Se evalúan k = 2, 3, 4, 5 y 6 con tres criterios combinados:

- **Codo (elbow):** costo intra-cluster de K-modes — buscar donde la reducción se aplana.
- **Silhouette:** cohesión interna y separación entre clusters (con distancia Hamming). Maximizar.
- **Davies-Bouldin:** relación dispersión intra-cluster vs distancia inter-cluster. Minimizar.

La decisión final combina los tres criterios con la **interpretabilidad pedagógica** de los perfiles.

In [ ]:
from kmodes.kmodes import KModes

costos   = {}
modelos  = {}

for k in range(2, 7):
    print(f'Entrenando k={k}...')
    km = KModes(n_clusters=k, init='Cao', n_init=10, verbose=0, random_state=42)
    km.fit(pdf_sample)
    costos[k]  = km.cost_
    modelos[k] = km
    print(f'  ✅ k={k} — costo: {km.cost_:,.0f}')

print('\nEntrenamiento completo.')

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import LabelEncoder
import numpy as np

## Calculo de metricas de evaluación

# Codificar categorías a enteros para sklearn
def encode_categorico(df):
    df_enc = df.copy()
    for col in df.columns:
        le = LabelEncoder()
        df_enc[col] = le.fit_transform(df[col].astype(str))
    return df_enc.values

# Subsample para silhouette y Davies-Bouldin (20k es suficiente)
np.random.seed(42)
idx_sub    = np.random.choice(len(pdf_sample), size=20_000, replace=False)
pdf_sub    = pdf_sample.iloc[idx_sub].reset_index(drop=True)
X_sub      = encode_categorico(pdf_sub)

silhouettes = {}
db_scores   = {}

for k, modelo in modelos.items():
    labels_sub       = modelo.predict(pdf_sub)
    silhouettes[k]   = silhouette_score(X_sub, labels_sub, metric='hamming')
    db_scores[k]     = davies_bouldin_score(X_sub.astype(float), labels_sub)

print(f'{"k":<5} {"Costo":>15} {"Silhouette":>12} {"Davies-Bouldin":>16}')
print('─' * 52)
for k in range(2, 7):
    print(f'{k:<5} {costos[k]:>15,.0f} {silhouettes[k]:>12.4f} {db_scores[k]:>16.4f}')

In [ ]:
import matplotlib.pyplot as plt

ks = list(range(2, 7))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Codo
axes[0].plot(ks, [costos[k] for k in ks], 'o-', color='steelblue', linewidth=2)
axes[0].set_title('Codo — Costo intra-cluster', fontweight='bold')
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Costo K-modes')
axes[0].set_xticks(ks)

# Silhouette
axes[1].plot(ks, [silhouettes[k] for k in ks], 'o-', color='seagreen', linewidth=2)
axes[1].set_title('Silhouette (↑ mejor)', fontweight='bold')
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silhouette promedio')
axes[1].set_xticks(ks)

# Davies-Bouldin
axes[2].plot(ks, [db_scores[k] for k in ks], 'o-', color='tomato', linewidth=2)
axes[2].set_title('Davies-Bouldin (↓ mejor)', fontweight='bold')
axes[2].set_xlabel('Número de clusters (k)')
axes[2].set_ylabel('Índice DB')
axes[2].set_xticks(ks)

plt.suptitle('Selección del número de clusters — K-modes ICFES', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Hallazgos — Selección del número de clusters

### Análisis de los criterios estadísticos

**Codo (costo intra-cluster)** 
Se identifican dos puntos de inflexión visibles: uno en **k=3**, donde la tasa de reducción comienza a desacelerarse notoriamente, y otro en **k=5**, donde la curva vuelve a aplanarse antes de continuar con una caída marginal hacia k=6. Esto sugiere que tanto k=3 como k=5 representan equilibrios eficientes entre complejidad del modelo y ganancia de cohesión intra-cluster.

**Silhouette**
El coeficiente de Silhouette decrece monotónicamente de k=2 a k=6, indicando que la separación relativa entre clusters disminuye a medida que se añaden grupos. Entre los valores de k ≥ 3, **k=3 obtiene el mejor Silhouette (~0.155)**, seguido de k=4 (~0.120). k=5 registra un valor de ~0.109, aceptable y coherente con la incorporación de perfiles más granulares.

**Davies-Bouldin**
k=4 es el peor valor del rango evaluado (pico ~5.6), descartándolo como opción. **k=5 obtiene el índice más bajo entre k ≥ 3 (~4.7)**, empatando prácticamente con k=3 (~4.6). Esto indica que los 5 clusters tienen una buena relación entre dispersión interna y separación entre grupos, comparable a la solución más parsimoniosa.

### Viabilidad de k=3 y k=5

Ambos valores son estadísticamente respaldados:

| Criterio | k=3 | k=5 |
|---|---|---|
| Codo | ✅ Inflexión clara | ✅ Segunda inflexión visible |
| Silhouette | ✅ Mejor entre k≥3 | ⚠️ Aceptable |
| Davies-Bouldin | ✅ Segundo mejor | ✅ Mejor entre k≥3 |
| Interpretabilidad | ⚠️ Perfiles muy amplios | ✅ Segmentación granular |

**k=3** representa la solución más parsimoniosa: clusters bien separados y cohesionados, pero con perfiles demasiado generales para diseñar intervenciones diferenciadas. En el contexto del ICFES, tres grupos tenderían a colapsar en categorías de alto, medio y bajo rendimiento, perdiendo matices socioeconómicos y de contexto escolar relevantes para la política educativa.

### Selección final: k=5

Se selecciona **k=5** por las siguientes razones:

1. **Respaldo estadístico dual:** es el único valor que figura entre los mejores en dos de los tres criterios (codo y Davies-Bouldin), mientras mantiene un Silhouette aceptable.
2. **Riqueza analítica:** cinco clusters permiten capturar combinaciones de rendimiento académico, contexto socioeconómico y tipo de institución que tres grupos no pueden distinguir. Por ejemplo, es posible diferenciar entre estudiantes de bajo rendimiento en colegios oficiales urbanos con acceso a internet, frente a estudiantes de bajo rendimiento en zonas rurales sin conectividad — perfiles que requieren intervenciones completamente distintas.
3. **Comunicabilidad:** cinco grupos son suficientemente pocos para ser comprendidos y apropiados por tomadores de decisión en política educativa, manteniendo la practicidad del análisis.

In [ ]:
# Modelo k=5 ya está entrenado — lo usamos directamente
modelo_final = modelos[5]

print('Centroides del modelo k=5:')
centroides = pd.DataFrame(modelo_final.cluster_centroids_, columns=pdf_sample.columns)
print(centroides.to_string())

# ── Asignar clusters al dataset completo ──────────────────────────────────────
print('\nConvirtiendo dfModel completo a pandas...')
pdf_full = dfModel.drop('ID_ESTUDIANTE').toPandas()
print(f'Registros totales: {len(pdf_full):,}')

print('Asignando clusters...')
etiquetas = modelo_final.predict(pdf_full)

# Agregar etiquetas al DataFrame de pandas
pdf_full['CLUSTER'] = etiquetas.astype(str)

# Recuperar ID_ESTUDIANTE y unir
ids = dfModel.select('ID_ESTUDIANTE').toPandas()
pdf_full['ID_ESTUDIANTE'] = ids['ID_ESTUDIANTE']

print('Distribución de clusters:')
print(pdf_full['CLUSTER'].value_counts().sort_index())

**Cluster 0** — 1,148,047 estudiantes (33.8%) — **"Conectado con rezago académico"**
Estrato 1, colegio oficial urbano, jornada mañana. Tiene internet y computador. Padres con bachillerato completo. Rendimiento mínimo en todas las áreas pero puntaje global medio-alto — el grupo más numeroso, urbano y con acceso tecnológico pero sin traducirlo en desempeño académico.

**Cluster 1** — 637,891 estudiantes (18.8%) — **"Rendimiento satisfactorio en lo público"**
Estrato 2, colegio oficial urbano con conectividad completa. Padres con bachillerato. Rendimiento satisfactorio en todas las áreas y puntaje global alto. Representa el segmento que logra buen desempeño dentro del sistema público.

**Cluster 2** — 717,996 estudiantes (21.2%) — **"Vulnerabilidad crítica"**
El grupo más desfavorecido. Estrato 1, hacinamiento alto, sin internet ni computador, padres con primaria incompleta. Rendimiento insuficiente en matemáticas, ciencias y sociales. Puntaje global bajo. Requiere intervención prioritaria e integral.

**Cluster 3** — 491,507 estudiantes (14.5%) — **"Exclusión digital con base familiar básica"**
Estrato 2, hacinamiento alto, sin conectividad. Padres con primaria completa — un escalón sobre el cluster 2. Rendimiento mínimo y puntaje medio-bajo. Brecha digital como principal barrera identificable.

**Cluster 4** — 397,308 estudiantes (11.7%) — **"Alta dotación académica y socioeconómica"**
El grupo más privilegiado. Estrato 3, colegio no oficial, jornada completa, padres con educación profesional completa. Rendimiento satisfactorio, inglés B1, puntaje global alto. Único cluster en colegio privado y con jornada completa.

---

Dos observaciones relevantes antes de continuar:

`AREA_COLEGIO` es `URBANO` en los 5 clusters — la distinción rural/urbana no emerge como diferenciadora en los centroides, seguramente por el desbalance dentro de los datos entre escuelas urbanas contra las rurales, aunque sí puede aparecer en el perfilamiento posterior. 

`NATURALEZA` es `OFICIAL` en 4 de 5 clusters — el sistema público domina la muestra y los perfiles.

In [ ]:
## Creación de DataFrame PySpark con los clusters

# Agregar PERIODO a dfModel para tener llave compuesta única
dfModelConPeriodo = dfModel.join(
    dfPy04.select('ID_ESTUDIANTE', 'PERIODO'),
    on='ID_ESTUDIANTE',
    how='left'
)

# UNA sola colección — garantiza alineación de filas
pdf_full = dfModelConPeriodo.toPandas()
print(f'Registros: {len(pdf_full):,}')

# Predecir sobre columnas del modelo únicamente
etiquetas = modelo_final.predict(pdf_full[MODEL_COLS])
pdf_full['CLUSTER'] = etiquetas.astype(str)

print('Distribución de clusters:')
print(pdf_full['CLUSTER'].value_counts().sort_index())

# ── Crear DataFrame Spark y unir con llave compuesta ─────────────────────────
dfClusters = sparkSigma.createDataFrame(
    pdf_full[['ID_ESTUDIANTE', 'PERIODO', 'CLUSTER']]
)

dfFinal = dfPy04.join(
    dfClusters,
    on=['ID_ESTUDIANTE', 'PERIODO'],
    how='left'
)

# Asignar nombres
dfFinal = dfFinal.withColumn(
    'CLUSTER_NOMBRE',
    F.when(F.col('CLUSTER') == '0', 'Conectado con rezago académico')
     .when(F.col('CLUSTER') == '1', 'Rendimiento satisfactorio en lo público')
     .when(F.col('CLUSTER') == '2', 'Vulnerabilidad crítica')
     .when(F.col('CLUSTER') == '3', 'Exclusión digital con base familiar básica')
     .when(F.col('CLUSTER') == '4', 'Alta dotación académica y socioeconómica')
)

print('dfFinal corregido. Registros:', dfFinal.count())

In [ ]:
ETIQUETAS_CLUSTER = {
    '0': 'Conectado con rezago académico',
    '1': 'Rendimiento satisfactorio en lo público',
    '2': 'Vulnerabilidad crítica',
    '3': 'Exclusión digital con base familiar básica',
    '4': 'Alta dotación académica y socioeconómica',
}

dfFinal = dfFinal.withColumn(
    'CLUSTER_NOMBRE',
    F.when(F.col('CLUSTER') == '0', ETIQUETAS_CLUSTER['0'])
     .when(F.col('CLUSTER') == '1', ETIQUETAS_CLUSTER['1'])
     .when(F.col('CLUSTER') == '2', ETIQUETAS_CLUSTER['2'])
     .when(F.col('CLUSTER') == '3', ETIQUETAS_CLUSTER['3'])
     .when(F.col('CLUSTER') == '4', ETIQUETAS_CLUSTER['4'])
)

dfFinal.groupBy('CLUSTER', 'CLUSTER_NOMBRE').count() \
       .orderBy('CLUSTER').show(truncate=False)

## Perfilamiento e interpretación de clusters

Se asignan etiquetas descriptivas a los 5 clusters identificados y se analiza su composición usando las variables de perfilamiento (no usadas en el modelo). El objetivo es enriquecer cada perfil con contexto geográfico, demográfico y de tendencia temporal.

| Cluster | Etiqueta | Estudiantes |
|---|---|---|
| 0 | Conectado con rezago académico | 1,148,047 |
| 1 | Rendimiento satisfactorio en lo público | 637,891 |
| 2 | Vulnerabilidad crítica | 717,996 |
| 3 | Exclusión digital con base familiar básica | 491,507 |
| 4 | Alta dotación académica y socioeconómica | 397,308 |

In [ ]:
# Verificar puntajes por cluster
dfFinal.groupBy('CLUSTER_NOMBRE').agg(
    F.round(F.avg('PUNT_GLOBAL'),  1).alias('GLOBAL'),
    F.round(F.avg('PUNT_MATE'),    1).alias('MATE'),
    F.round(F.avg('PUNT_LECTURA'), 1).alias('LECTURA'),
    F.round(F.avg('PUNT_CIENCIAS'),1).alias('CIENCIAS'),
    F.round(F.avg('PUNT_SOCIALES'),1).alias('SOCIALES'),
    F.round(F.avg('PUNT_INGLES'),  1).alias('INGLES'),
).orderBy('GLOBAL').show(truncate=False)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

## Genero por cluster

colores = ['#4C72B0', '#55A868', '#C44E52', '#DD8452', '#8172B2']
nombres_cortos = {
    '0': 'Conectado\nrezago',
    '1': 'Satisfactorio\npúblico',
    '2': 'Vulnerabilidad\ncrítica',
    '3': 'Exclusión\ndigital',
    '4': 'Alta\ndotación',
}
orden = ['0','1','2','3','4']

pdf_genero = dfFinal.groupBy('CLUSTER','GENERO').count() \
                    .toPandas().pivot(index='CLUSTER', columns='GENERO', values='count').fillna(0)

total_gen = pdf_genero.sum(axis=1)
pdf_genero_pct = pdf_genero.div(total_gen, axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
etiq = [nombres_cortos[c] for c in orden]
ax.bar(etiq, pdf_genero_pct.loc[orden, 'F'], color='#DD8452', label='Femenino', alpha=0.85)
ax.bar(etiq, pdf_genero_pct.loc[orden, 'M'],
       bottom=pdf_genero_pct.loc[orden, 'F'], color='#4C72B0', label='Masculino', alpha=0.85)
for i, c in enumerate(orden):
    pct_f = pdf_genero_pct.loc[c, 'F']
    pct_m = pdf_genero_pct.loc[c, 'M']
    ax.text(i, pct_f / 2, f'{pct_f:.1f}%', ha='center', va='center',
            color='white', fontweight='bold', fontsize=9)
    ax.text(i, pct_f + pct_m / 2, f'{pct_m:.1f}%', ha='center', va='center',
            color='white', fontweight='bold', fontsize=9)
ax.set_title('Distribución por género por cluster', fontweight='bold', fontsize=13)
ax.set_ylabel('Porcentaje (%)')
ax.set_ylim(0, 110)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
pdf_area = dfFinal.groupBy('CLUSTER','AREA_COLEGIO').count() \
                  .toPandas().pivot(index='CLUSTER', columns='AREA_COLEGIO', values='count').fillna(0)

total_area = pdf_area.sum(axis=1)
pdf_area_pct = pdf_area.div(total_area, axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(etiq, pdf_area_pct.loc[orden, 'RURAL'],  color='#55A868', label='Rural',  alpha=0.85)
ax.bar(etiq, pdf_area_pct.loc[orden, 'URBANO'],
       bottom=pdf_area_pct.loc[orden, 'RURAL'], color='#4C72B0', label='Urbano', alpha=0.85)
for i, c in enumerate(orden):
    pct_r = pdf_area_pct.loc[c, 'RURAL']
    pct_u = pdf_area_pct.loc[c, 'URBANO']
    if pct_r > 3:
        ax.text(i, pct_r / 2, f'{pct_r:.1f}%', ha='center', va='center',
                color='white', fontweight='bold', fontsize=9)
    ax.text(i, pct_r + pct_u / 2, f'{pct_u:.1f}%', ha='center', va='center',
            color='white', fontweight='bold', fontsize=9)
ax.set_title('Distribución rural / urbano por cluster', fontweight='bold', fontsize=13)
ax.set_ylabel('Porcentaje (%)')
ax.set_ylim(0, 110)
ax.legend()
plt.tight_layout()
plt.savefig('/Almacen/jupyterhub/icfes_spark/data/perfil_area.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
pdf_depto = dfFinal.groupBy('CLUSTER', 'DEPTO_COLEGIO').count() \
                   .orderBy('CLUSTER', F.col('count').desc()) \
                   .toPandas()

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for i, c in enumerate(orden):
    top5 = pdf_depto[pdf_depto['CLUSTER'] == c].head(5)
    axes[i].barh(top5['DEPTO_COLEGIO'], top5['count'], color=colores[i], alpha=0.85)
    axes[i].set_title(nombres_cortos[c], fontweight='bold', fontsize=9)
    axes[i].set_xlabel('Estudiantes')
    axes[i].invert_yaxis()
    axes[i].xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}k'))

plt.suptitle('Top 5 departamentos por cluster', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/Almacen/jupyterhub/icfes_spark/data/perfil_departamentos.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
pdf_auto = dfFinal.groupBy('CLUSTER','TIENE_AUTOMOVIL').count() \
                  .toPandas().pivot(index='CLUSTER', columns='TIENE_AUTOMOVIL', values='count').fillna(0)

pdf_lav = dfFinal.groupBy('CLUSTER','TIENE_LAVADORA').count() \
                 .toPandas().pivot(index='CLUSTER', columns='TIENE_LAVADORA', values='count').fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for pdf, ax, titulo in [
    (pdf_auto, axes[0], 'Tiene automóvil'),
    (pdf_lav,  axes[1], 'Tiene lavadora'),
]:
    total = pdf.sum(axis=1)
    pct_si = pdf.get('SI', pd.Series(0, index=pdf.index)).div(total) * 100
    ax.bar(etiq, pct_si.loc[orden], color=colores, alpha=0.85)
    for j, c in enumerate(orden):
        ax.text(j, pct_si.loc[c] + 1, f'{pct_si.loc[c]:.1f}%',
                ha='center', fontsize=9, fontweight='bold')
    ax.set_title(f'% con {titulo} por cluster', fontweight='bold', fontsize=12)
    ax.set_ylabel('% que responde SÍ')
    ax.set_ylim(0, 100)

plt.suptitle('Recursos del hogar adicionales por cluster', fontweight='bold',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/Almacen/jupyterhub/icfes_spark/data/perfil_recursos_hogar.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
pdf_tiempo = dfFinal.groupBy('CLUSTER', 'PERIODO').count().toPandas()
pdf_tiempo['ANIO'] = pdf_tiempo['PERIODO'].astype(str).str[:4].astype(int)
pdf_tiempo = pdf_tiempo.groupby(['CLUSTER','ANIO'])['count'].sum().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
for i, c in enumerate(orden):
    sub = pdf_tiempo[pdf_tiempo['CLUSTER'] == c].sort_values('ANIO')
    ax.plot(sub['ANIO'], sub['count'], 'o-', color=colores[i],
            label=nombres_cortos[c].replace('\n',' '), linewidth=2)

ax.set_title('Evolución del tamaño de clusters por año', fontweight='bold', fontsize=13)
ax.set_xlabel('Año')
ax.set_ylabel('Número de estudiantes')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('/Almacen/jupyterhub/icfes_spark/data/perfil_temporal.png',
            dpi=150, bbox_inches='tight')
plt.show()